# Notebook for creating plots of cluster summaries
Needs to be run after `..._clustering.ipynb`

In [ ]:
# Update path and working directory with project root
import os
import sys
from pathlib import Path

project_path = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file())
os.chdir(project_path)
sys.path.insert(0, str(project_path))

In [ ]:
# Imports
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import plotnine as p9
from IPython.display import display

from utils.clustering_utils import cluster_summaries
from utils.misc_utils import display_side_by_side, value_counts
from utils.plotting_utils import (
    make_plot_categorical_var_dist_by_cluster,
    make_plot_continuous_var_dist_by_cluster,
    make_plot_histogram_by_cluster,
)
from utils.saving_utils import DatasetBundle, save_plot

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

In [ ]:
# Load data
data_discov_full = DatasetBundle.load(Path("data/preprocessed/data_discov_full"))
assert data_discov_full.meta is not None
assert data_discov_full.na is not None
data_valid_full = DatasetBundle.load(Path("data/preprocessed/data_valid_full"))
assert data_valid_full.meta is not None
assert data_valid_full.na is not None

metrics_grid_search_discov = pd.read_parquet(Path("outputs/gmm_clustering/metrics_grid_search_discov.parquet"))

proba_and_labels_discov = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_discov.parquet"))
proba_and_labels_valid = pd.read_parquet(Path("outputs/gmm_clustering/cluster_pred_valid.parquet"))

# Set outputs folder
output_folder = Path("outputs/gmm_clustering")
output_folder.mkdir(exist_ok=True)

# Set predicted probability and labels
proba_discov = proba_and_labels_discov.drop("label", axis="columns")
proba_valid = proba_and_labels_valid.drop("label", axis="columns")
labels_discov = proba_and_labels_discov["label"]
labels_valid = proba_and_labels_valid["label"]

# Set sample weights
sample_weight_discov = data_discov_full.data["wichtn"].to_numpy()
sample_weight_valid = data_valid_full.data["wichtn"].to_numpy()

In [ ]:
# Print cluster proportions
proportions_discov = (
    value_counts(labels_discov, sort=True, normalize=True, dropna=False, sample_weight=sample_weight_discov)
    .apply(lambda x: f"{x:.2%}")
    .to_frame()
    .reset_index(names="cluster")
)
proportions_valid = (
    value_counts(labels_valid, sort=True, normalize=True, dropna=False, sample_weight=sample_weight_valid)
    .apply(lambda x: f"{x:.2%}")
    .to_frame()
    .reset_index(names="cluster")
)
display_side_by_side(proportions_discov, proportions_valid, captions=["discov", "valid"])

In [ ]:
# Interpretation of variables in the discovery dataset
cluster_summaries_not_nominal_discov, cluster_summaries_nominal_discov = cluster_summaries(
    data_discov_full.drop(["lfd", "fall"]), labels_discov, sample_weight=sample_weight_discov
)

print("Summaries of continuous, binary, and ordinal variables in the discovery dataset:")
display(cluster_summaries_not_nominal_discov)

print("Summaries of nominal variables in the discovery dataset:")
display(cluster_summaries_nominal_discov)

In [ ]:
# Interpretation of variables in the validation dataset
cluster_summaries_not_nominal_valid, cluster_summaries_nominal_valid = cluster_summaries(
    data_valid_full.drop(["lfd", "fall"]), labels_valid, sample_weight=sample_weight_valid
)

print("Summaries of continuous, binary, and ordinal variables in the validation dataset:")
display(cluster_summaries_not_nominal_valid)

print("Summaries of nominal variables in the validation dataset:")
display(cluster_summaries_nominal_valid)

In [ ]:
# Plot distribution of continuous variable in different clusters
var = "Anzahl"
quantiles = [0.02, 0.1, 0.33, 0.5, 0.67, 0.9, 0.98]

# Make plots
continuous_var_plot_discov = make_plot_continuous_var_dist_by_cluster(
    data_discov_full.data[var],
    labels_discov,
    quantiles=quantiles,
    sample_weight=sample_weight_discov,
    sort=True,
    ascending=False,
    cluster_size_decimals=1,
    column_size=7,
)
continuous_var_plot_valid = make_plot_continuous_var_dist_by_cluster(
    data_valid_full.data[var],
    labels_valid,
    quantiles=quantiles,
    sample_weight=sample_weight_valid,
    sort=True,
    ascending=False,
    cluster_size_decimals=1,
    column_size=7,
)

# Combine plots
continuous_var_plot_all = (
    continuous_var_plot_discov + p9.labs(title="Discovery") + continuous_var_plot_valid + p9.labs(title="Validation")
) & p9.theme(figure_size=(16, 6))
continuous_var_plot_all.show()

# Save plots
save_plot(continuous_var_plot_discov, output_folder / "continuous_var_plot_discov.png", dpi=150)
save_plot(continuous_var_plot_valid, output_folder / "continuous_var_plot_valid.png", dpi=150)

# save_plot(continuous_var_plot_all, output_folder / "continuous_var_plot_all.png", dpi=150)

In [ ]:
# Plot distribution of categorical variable in different clusters
var = "erwerb"
var_value_labels = ast.literal_eval(data_discov_full.meta.loc[lambda x: x["name"].eq(var), "labels"].item())

# Make plots
categorical_var_plot_discov = make_plot_categorical_var_dist_by_cluster(
    data_discov_full.data[var],
    labels_discov,
    sample_weight=sample_weight_discov,
    cluster_size_decimals=1,
    value_labels=var_value_labels,
)
categorical_var_plot_valid = make_plot_categorical_var_dist_by_cluster(
    data_valid_full.data[var],
    labels_valid,
    sample_weight=sample_weight_valid,
    cluster_size_decimals=1,
    value_labels=var_value_labels,
)

# Combine plots
categorical_var_plot_all = (
    categorical_var_plot_discov + p9.labs(title="Discovery") + categorical_var_plot_valid + p9.labs(title="Validation")
) & p9.theme(figure_size=(16, 6))
categorical_var_plot_all.show()

# Save plots
save_plot(categorical_var_plot_discov, output_folder / "categorical_var_plot_discov.png", dpi=150)
save_plot(categorical_var_plot_valid, output_folder / "categorical_var_plot_valid.png", dpi=150)

# save_plot(categorical_var_plot_all, output_folder / "categorical_var_plot_all.png", dpi=150)

In [ ]:
# Save summaries
proportions_discov.to_parquet(output_folder / "proportions_discov.parquet", index=False)
proportions_valid.to_parquet(output_folder / "proportions_valid.parquet", index=False)

cluster_summaries_not_nominal_discov.to_parquet(
    output_folder / "cluster_discov_summaries_not_nominal.parquet", index=False
)
cluster_summaries_nominal_discov.to_parquet(
    output_folder / "cluster_discov_summaries_nominal.parquet", index=False
)
cluster_summaries_not_nominal_valid.to_parquet(
    output_folder / "cluster_valid_summaries_not_nominal.parquet", index=False
)
cluster_summaries_nominal_valid.to_parquet(output_folder / "cluster_valid_summaries_nominal.parquet", index=False)

In [ ]:
# Plot the distributions of all variables (except "lfd", "fall", and "bikgk10")
n_clusters: int | None = 5  # Set to None to plot all clusters
if n_clusters is not None and n_clusters < 1:
    raise ValueError("n_clusters must be positive or None.")

# Use the existing cluster order: 0 is largest, 1 is second largest, and so on
clusters_discov = labels_discov.cat.categories[:n_clusters]
clusters_valid = labels_valid.cat.categories[:n_clusters]

all_vars_dists_folder = output_folder / "all_vars_dists"
all_vars_dists_folder.mkdir(exist_ok=True)

def _save_var_dist_plot(var: str) -> None:
    discov_row = data_discov_full.meta[lambda x: x["name"].eq(var)].iloc[0]  # type: ignore
    valid_row = data_valid_full.meta[lambda x: x["name"].eq(var)].iloc[0]  # type: ignore
    var_group = discov_row["group"]
    var_type = discov_row["type"]
    var_group_folder = all_vars_dists_folder / var_group
    var_group_folder.mkdir(exist_ok=True)

    if var_type == "continuous":
        values = pd.concat(
            [data_discov_full.data[var], data_valid_full.data[var]], ignore_index=True
        ).dropna()
        values = values[np.isfinite(values)]
        bins = np.histogram_bin_edges(values, bins=15)

        # Make plots
        continuous_var_plot_discov_weighted = make_plot_histogram_by_cluster(
            data_discov_full.data[var],
            labels_discov,
            clusters=clusters_discov,
            bins=bins,
            sample_weight=sample_weight_discov,
            cluster_size_decimals=1,
        )
        continuous_var_plot_discov = make_plot_histogram_by_cluster(
            data_discov_full.data[var],
            labels_discov,
            clusters=clusters_discov,
            bins=bins,
            sample_weight=None,
            cluster_size_decimals=1,
        )
        continuous_var_plot_valid_weighted = make_plot_histogram_by_cluster(
            data_valid_full.data[var],
            labels_valid,
            clusters=clusters_valid,
            bins=bins,
            sample_weight=sample_weight_valid,
            cluster_size_decimals=1,
        )
        continuous_var_plot_valid = make_plot_histogram_by_cluster(
            data_valid_full.data[var],
            labels_valid,
            clusters=clusters_valid,
            bins=bins,
            sample_weight=None,
            cluster_size_decimals=1,
        )

        # Combine plots
        continuous_var_plot_all = (
            (
                continuous_var_plot_discov_weighted + p9.labs(title="Discovery weighted")
                | continuous_var_plot_discov + p9.labs(title="Discovery")
            )
            / (
                continuous_var_plot_valid_weighted + p9.labs(title="Validation weighted")
                | continuous_var_plot_valid + p9.labs(title="Validation")
            )
        ) & p9.theme(figure_size=(16, 10))

        save_plot(continuous_var_plot_all, var_group_folder / f"{var_type}_{var}_plot.png", dpi=150)
        return

    if var_type in ["binary", "nominal", "ordinal"]:
        # Make plots
        categorical_var_plot_discov_weighted = make_plot_categorical_var_dist_by_cluster(
            data_discov_full.data[var],
            labels_discov,
            clusters=clusters_discov,
            sample_weight=sample_weight_discov,
            cluster_size_decimals=1,
            value_labels=ast.literal_eval(discov_row["labels"]),
        )
        categorical_var_plot_discov = make_plot_categorical_var_dist_by_cluster(
            data_discov_full.data[var],
            labels_discov,
            clusters=clusters_discov,
            sample_weight=None,
            cluster_size_decimals=1,
            value_labels=ast.literal_eval(discov_row["labels"]),
        )
        categorical_var_plot_valid_weighted = make_plot_categorical_var_dist_by_cluster(
            data_valid_full.data[var],
            labels_valid,
            clusters=clusters_valid,
            sample_weight=sample_weight_valid,
            cluster_size_decimals=1,
            value_labels=ast.literal_eval(valid_row["labels"]),
        )
        categorical_var_plot_valid = make_plot_categorical_var_dist_by_cluster(
            data_valid_full.data[var],
            labels_valid,
            clusters=clusters_valid,
            sample_weight=None,
            cluster_size_decimals=1,
            value_labels=ast.literal_eval(valid_row["labels"]),
        )

        # Combine plots
        categorical_var_plot_all = (
            (
                categorical_var_plot_discov_weighted + p9.labs(title="Discovery weighted")
                | categorical_var_plot_discov + p9.labs(title="Discovery")
            )
            / (
                categorical_var_plot_valid_weighted + p9.labs(title="Validation weighted")
                | categorical_var_plot_valid + p9.labs(title="Validation")
            )
        ) & p9.theme(figure_size=(16, 10))

        save_plot(categorical_var_plot_all, var_group_folder / f"{var_type}_{var}_plot.png", dpi=150)


for var in data_discov_full.drop(["lfd", "fall", "bikgk10"]).meta["name"]:  # type: ignore
    _save_var_dist_plot(var)